# Image Provenance Tracking System - Complete Tutorial

This notebook demonstrates the complete workflow of the Image Provenance Tracking System.

## What you'll learn:
1. Upload and track images
2. Record distribution events (hops)
3. Detect AI derivatives
4. Re-entry detection
5. Lineage visualization

## Setup

First, import the SDK and create a client:

In [ ]:
from sdk import ImageProvenanceClient
from PIL import Image
import io
import json

# Create client
client = ImageProvenanceClient('http://localhost:8000')

# Check connection
health = client.health_check()
print(f"✓ Connected to API (status: {health['status']})")

## Step 1: Upload an Image

Upload an image and register it in the tracking system:

In [ ]:
# Create a test image
test_image = Image.new('RGB', (800, 600), color='blue')
test_image.save('test_image.png')

# Upload to system
upload_result = client.upload_image(
    image_path='test_image.png',
    user_id='demo_user_123',
    device_id='device_laptop_1'
)

image_id = upload_result['image_id']
watermark_id = upload_result['watermark_id']

print(f"✓ Image uploaded successfully!")
print(f"  Image ID: {image_id}")
print(f"  Watermark ID: {watermark_id}")
print(f"\nFingerprints:")
for key, value in upload_result['fingerprints'].items():
    print(f"  {key}: {value}")

## Step 2: Record Distribution Events

Track how the image moves between devices:

In [ ]:
# Share from laptop to mobile
share1 = client.record_share(
    image_id=image_id,
    device_id='device_mobile_1',
    platform='mobile_app'
)

print(f"✓ Hop 1: laptop → mobile (Event: {share1['event_id']})")

# Share from mobile to web
share2 = client.record_share(
    image_id=image_id,
    device_id='device_web_1',
    platform='web',
    source_event_id=share1['event_id']
)

print(f"✓ Hop 2: mobile → web (Event: {share2['event_id']})")

# Share from web to another device
share3 = client.record_share(
    image_id=image_id,
    device_id='device_tablet_1',
    platform='tablet',
    source_event_id=share2['event_id']
)

print(f"✓ Hop 3: web → tablet (Event: {share3['event_id']})")
print(f"\n✓ Distribution chain recorded: 3 hops across 4 devices")

## Step 3: View Complete Lineage

Retrieve and visualize the complete hop chain:

In [ ]:
# Get lineage
lineage = client.get_lineage(image_id)

print("Image Lineage:")
print(f"  Original: {lineage['original_image']['filename']}")
print(f"  Created: {lineage['original_image']['created_at']}")
print(f"\nStatistics:")
stats = lineage['statistics']
print(f"  Total events: {stats['total_events']}")
print(f"  Unique devices: {stats['unique_devices']}")
print(f"  Max hop depth: {stats['max_hop_depth']}")
print(f"\nEvent breakdown:")
for event_type, count in stats['event_breakdown'].items():
    print(f"  {event_type}: {count}")

## Step 4: Simulate AI Derivative Creation

Create a modified version (simulating AI generation):

In [ ]:
# Create a modified version (lighter blue)
derivative_image = Image.new('RGB', (800, 600), color='lightblue')
derivative_image.save('derivative_image.png')

# Upload derivative as new image
derivative_result = client.upload_image(
    image_path='derivative_image.png',
    user_id='ai_generator_service'
)

derivative_id = derivative_result['image_id']

print(f"✓ Derivative created: {derivative_id}")
print(f"  (In production, the system would auto-detect the similarity)")

## Step 5: Re-Entry Detection

Detect when the original image re-appears:

In [ ]:
# Detect the original image (simulating re-entry)
detection = client.detect_image(
    image_path='test_image.png',
    device_id='device_new_unknown'
)

if detection['detected']:
    print("✓ IMAGE DETECTED!")
    print(f"  Confidence: {detection['confidence']}")
    print(f"  Detection methods: {', '.join(detection['methods'])}")
    
    print(f"\n  Matches found:")
    for match in detection['matches']:
        print(f"    • Method: {match.get('method')}")
        if 'similarity' in match:
            print(f"      Similarity: {match['similarity']:.2%}")
    
    if detection.get('lineage'):
        print(f"\n  Complete lineage retrieved:")
        stats = detection['lineage']['statistics']
        print(f"    Hops: {stats['total_events']}")
        print(f"    Devices: {stats['unique_devices']}")
else:
    print("✗ No match found")

## Step 6: Query Derivatives

Find all AI derivatives of the original:

In [ ]:
# Get derivatives
derivatives = client.get_derivatives(
    image_id=image_id,
    similarity_threshold=0.70
)

print(f"Found {len(derivatives)} derivative(s):\n")

for deriv in derivatives:
    print(f"  Derivative ID: {deriv['derivative_id']}")
    print(f"  Similarity: {deriv['similarity']:.2%}")
    print(f"  Detection method: {deriv['detection_method']}")
    print(f"  AI model: {deriv.get('ai_model', 'N/A')}")
    print()

## Step 7: System Statistics

View overall system statistics:

In [ ]:
# Get detection statistics
stats = client.get_detection_stats(days=30)

print("System Statistics (Last 30 days):")
print(f"  Total detections: {stats.get('total_detections', 0)}")
print(f"  Unique images detected: {stats.get('unique_images_detected', 0)}")

if stats.get('detection_methods'):
    print(f"\n  Detection methods breakdown:")
    for method, data in stats['detection_methods'].items():
        print(f"    {method}: {data['count']} detections")

## Complete Workflow Summary

You've successfully:

1. ✅ Uploaded an image with automatic fingerprinting and watermarking
2. ✅ Tracked distribution across multiple devices (hops)
3. ✅ Retrieved complete lineage with statistics
4. ✅ Created AI derivatives
5. ✅ Detected re-entry with multi-method verification
6. ✅ Queried derivatives by similarity
7. ✅ Viewed system statistics

## Next Steps

- Explore the API documentation at http://localhost:8000/docs
- Try the CLI tool: `python cli.py --help`
- View the dashboard at http://localhost:8000/dashboard.html
- Read the README.md for advanced usage